<a href="https://colab.research.google.com/github/AnnaNaumchuk/AB-Testing---SQL-for-analysis/blob/main/Project_2_A_B_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Встановлюємо бібліотеку та робимо аутентифікацію до файлів**

In [ ]:
! pip install google-cloud-bigquery

#імпортуємо модулі
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np

#Аутентифікація
auth.authenticate_user()

client = bigquery.Client(project="data-analytics-mate")

**Основний запит**

In [ ]:
query = """

# AB Testing - SQL for analysis

with session_info as(
 select
    s.date,
    s.ga_session_id,
    sp.country,
    sp.device,
    sp.continent,
    sp.channel,
    ab.test,
    ab.test_group
 from `data-analytics-mate.DA.ab_test` ab
 join `data-analytics-mate.DA.session` s
 on ab.ga_session_id = s.ga_session_id
 join `data-analytics-mate.DA.session_params` sp
 on sp.ga_session_id = ab.ga_session_id
),
session_with_orders as (
 select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count( distinct o.ga_session_id) as session_with_orders
 from `DA.order` o
 join session_info
 on o.ga_session_id = session_info.ga_session_id
 group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
events as (
 select
   session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name,
    count( sp.ga_session_id) as event_cnt
 from `DA.event_params` sp
 join session_info
 on sp.ga_session_id = session_info.ga_session_id
 group by
   session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name
),
session as (
 SELECT
   session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct session_info.ga_session_id) as session_cnt
 from session_info
 group by
   session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
account as (
  SELECT
   session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct acs.ga_session_id) as new_account_cnt
  from `DA.account_session`acs
  join session_info
  on acs.ga_session_id = session_info.ga_session_id
  group by    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
)
SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session_with_orders' as event_name,
  session_with_orders.session_with_orders as value
from session_with_orders
union all
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  event_cnt as value
from events
union all
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' as event_name,
  session_cnt as value
from session
union all
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' as event_name,
  new_account_cnt as value
from account

"""


#Виконання запиту
query_job = client.query(query)
results = query_job.result()


# Перетворення результатів у DataFrame
df = results.to_dataframe(create_bqstorage_client=False)


In [ ]:
# Виводимо таблицю
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new account,1


In [ ]:
# Переглядаємо інформацію по таблиці
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


In [ ]:
# Шукаємо дублікати
print(df.isnull().sum())

date          0
country       0
device        0
continent     0
channel       0
test          0
test_group    0
event_name    0
value         0
dtype: int64


**Створюємо метрики та таблицю з загальними результатами**

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
metrics = [{"name": "begin_checkout/session", "num": "begin_checkout", "den": "session"},
           {"name": "add_payment_info/session", "num": "add_payment_info", "den": "session"},
           {"name": "add_shipping_info/session", "num": "add_shipping_info", "den": "session"},
           {"name": "new account/session", "num": "new account", "den": "session"}]
results = []
def make_pivot(df):
        agg = df.groupby(["test", "test_group", "event_name"])["value"].sum().reset_index()

        pivot = agg.pivot_table(index=["test", "test_group"],
        columns="event_name", values="value", fill_value=0).reset_index()

        pivot.columns.name = None
        return pivot
pivot_total = make_pivot(df)
pivot_total.head()

,test,test_group,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,new account,page_view,...,select_item,select_promotion,session,session_start,session_with_orders,user_engagement,view_item,view_item_list,view_promotion,view_search_results
0,1,1,1988.0,3034.0,1395.0,3784.0,368.0,30596.0,3823.0,191543.0,...,543.0,1275.0,45362.0,45905.0,4514.0,171788.0,62335.0,27.0,29188.0,3678.0
1,1,2,2229.0,3221.0,1366.0,4021.0,353.0,30512.0,3681.0,198050.0,...,530.0,1323.0,45193.0,45649.0,4526.0,179081.0,65337.0,24.0,29117.0,3882.0
2,2,1,2344.0,3480.0,2811.0,4262.0,337.0,34511.0,4165.0,220275.0,...,905.0,1477.0,50637.0,51219.0,5102.0,198266.0,72717.0,24.0,32367.0,4282.0
3,2,2,2409.0,3510.0,3061.0,4313.0,413.0,34171.0,4184.0,212320.0,...,946.0,1406.0,50244.0,50808.0,5003.0,189931.0,68700.0,29.0,31680.0,4198.0
4,3,1,3623.0,5298.0,17674.0,9532.0,280.0,50438.0,5856.0,286351.0,...,8735.0,2020.0,70047.0,71312.0,6951.0,249921.0,93931.0,9.0,41169.0,5764.0


**Проводимо тест для загальних результатів**

In [ ]:
tests = pivot_total['test'].unique()

for test_id in tests:
    df_test = pivot_total[pivot_total['test'] == test_id]
    group1 = df_test[df_test['test_group'] == 1]
    group2 = df_test[df_test['test_group'] == 2]

    for metric in metrics:
        num1 = group1[metric['num']].sum()
        num2 = group2[metric['num']].sum()
        den1 = group1[metric['den']].sum()
        den2 = group2[metric['den']].sum()

        conv1 = num1 / den1 if den1 > 0 else 0
        conv2 = num2 / den2 if den2 > 0 else 0

        # нова метрика
        metric_change = (conv2 - conv1) / conv1 if conv1 > 0 else 0

        count = np.array([num1, num2])
        nobs = np.array([den1, den2])
        z_stat, p_value = proportions_ztest(count, nobs)

        significant = p_value < 0.05
        print(f"Metric {metric['name']}, p-value = {p_value:.4f}, z_stat = {z_stat:.4f}, change = {metric_change:.2%}")
        results.append({
            "test_number": test_id,
            "metrics": metric['name'],
            "numerator_ev": num1,
            "denominator_ev": den1,
            "numerator_cohort": num2,
            "denominator_cohort": den2,
            "conversion_rate_1": conv1,
            "conversion_rate_2": conv2,
            "metric_change": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": significant
        })


Metric begin_checkout/session, p-value = 0.0029, z_stat = -2.9788, change = 6.66%
Metric add_payment_info/session, p-value = 0.0001, z_stat = -3.9249, change = 12.54%
Metric add_shipping_info/session, p-value = 0.0092, z_stat = -2.6036, change = 6.56%
Metric new account/session, p-value = 0.1229, z_stat = 1.5429, change = -3.35%
Metric begin_checkout/session, p-value = 0.3406, z_stat = -0.9529, change = 1.99%
Metric add_payment_info/session, p-value = 0.2146, z_stat = -1.2410, change = 3.58%
Metric add_shipping_info/session, p-value = 0.4780, z_stat = -0.7096, change = 1.65%
Metric new account/session, p-value = 0.5560, z_stat = -0.5888, change = 1.24%
Metric begin_checkout/session, p-value = 0.0120, z_stat = 2.5114, change = -3.35%
Metric add_payment_info/session, p-value = 0.5201, z_stat = -0.6432, change = 1.47%
Metric add_shipping_info/session, p-value = 0.1574, z_stat = 1.4137, change = -2.62%
Metric new account/session, p-value = 0.5199, z_stat = 0.6435, change = -1.13%
Metric be

**Таблиця з континентами**

In [ ]:
def make_pivot_continent(df):
    agg = df.groupby(["test", "test_group", "continent", "event_name"])["value"].sum().reset_index()

    pivot = agg.pivot_table(index=["test", "test_group", "continent"], columns="event_name", values="value", fill_value=0).reset_index()
    pivot.columns.name = None
    return pivot
pivot_continent = make_pivot_continent(df)
pivot_continent.head()

,test,test_group,continent,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,new account,...,select_item,select_promotion,session,session_start,session_with_orders,user_engagement,view_item,view_item_list,view_promotion,view_search_results
0,1,1,(not set),7.0,4.0,3.0,4.0,1.0,65.0,7.0,...,0.0,1.0,97.0,98.0,11.0,236.0,71.0,0.0,50.0,5.0
1,1,1,Africa,19.0,41.0,16.0,54.0,2.0,325.0,36.0,...,6.0,20.0,494.0,496.0,46.0,1885.0,618.0,0.0,364.0,25.0
2,1,1,Americas,1107.0,1704.0,838.0,2130.0,213.0,16879.0,2165.0,...,290.0,674.0,25164.0,25452.0,2505.0,96138.0,35351.0,19.0,16060.0,2131.0
3,1,1,Asia,512.0,730.0,299.0,915.0,77.0,7220.0,878.0,...,156.0,302.0,10626.0,10782.0,1072.0,39641.0,14357.0,4.0,6797.0,886.0
4,1,1,Europe,324.0,528.0,214.0,645.0,71.0,5789.0,700.0,...,91.0,261.0,8472.0,8572.0,828.0,31962.0,11214.0,4.0,5605.0,597.0


**Тест пропорцій для континентів**

In [ ]:
tests1 = pivot_continent['test'].unique()

for test_id in tests1:
    df_test1 = pivot_continent[pivot_continent['test'] == test_id]
    group1 =  df_test1[df_test1['test_group'] == 1]
    group2 =  df_test1[df_test1['test_group'] == 2]

    for metric in metrics:
        num1 = group1[metric['num']].sum()
        num2 = group2[metric['num']].sum()
        den1 = group1[metric['den']].sum()
        den2 = group2[metric['den']].sum()

        conv1 = num1 / den1 if den1 > 0 else 0
        conv2 = num2 / den2 if den2 > 0 else 0

        count = np.array([num1, num2])
        nobs = np.array([den1, den2])
        z_stat, p_value = proportions_ztest(count, nobs)

        metric_change = conv2 / conv1 if conv1 > 0 else np.nan
        significant = p_value < 0.05
        print(f"Metric {metric['name']}, p-value = {p_value:.4f}, z_stat = {z_stat:.4f}")

Metric begin_checkout/session, p-value = 0.0029, z_stat = -2.9788
Metric add_payment_info/session, p-value = 0.0001, z_stat = -3.9249
Metric add_shipping_info/session, p-value = 0.0092, z_stat = -2.6036
Metric new account/session, p-value = 0.1229, z_stat = 1.5429
Metric begin_checkout/session, p-value = 0.3406, z_stat = -0.9529
Metric add_payment_info/session, p-value = 0.2146, z_stat = -1.2410
Metric add_shipping_info/session, p-value = 0.4780, z_stat = -0.7096
Metric new account/session, p-value = 0.5560, z_stat = -0.5888
Metric begin_checkout/session, p-value = 0.0120, z_stat = 2.5114
Metric add_payment_info/session, p-value = 0.5201, z_stat = -0.6432
Metric add_shipping_info/session, p-value = 0.1574, z_stat = 1.4137
Metric new account/session, p-value = 0.5199, z_stat = 0.6435
Metric begin_checkout/session, p-value = 0.0459, z_stat = 1.9960
Metric add_payment_info/session, p-value = 0.1162, z_stat = 1.5711
Metric add_shipping_info/session, p-value = 0.0741, z_stat = 1.7858
Metric

Посилання на таблицю                  
https://docs.google.com/spreadsheets/d/14Z2vowx0QHcHHMPxHiH2x4CePjgO4jGbjz0V263hr9I/edit?usp=sharing


Посилання на дешборд                                
https://public.tableau.com/views/ABtestingPart2/ABtest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link